# Motorlu Taşıt Sigortası — Satış Kanalı Performans İzleme ve Erken Uyarı Sistemi

## Proje Özeti

Bu proje, gerçek bir motorlu taşıt sigortası poliçe portföyü üzerinde uçtan uca
bir analitik sistem geliştirmeyi amaçlamaktadır: veri temizleme, KPI hesaplama,
istatistiksel anomali tespiti, müşteri segmentasyonu ve bunların hepsini
birleştiren periyodik/otomatik bir raporlama pipeline'ı.

Kullanılan veri seti, 105.555 satırlık bir motorlu taşıt sigortası poliçe
portföyüdür (Mendeley Data, "Motor vehicle insurance data"). Veri panel
yapısındadır: her poliçe sahibi (ID), 2015-2018 arası birden fazla yıllık
yenileme kaydına sahiptir.

Projenin amacı yalnızca metrik hesaplamak değil, **her yöntem seçiminin veriden
gelen kanıtla desteklenmesini** sağlamaktır — bu notebook, kullanılan her
yöntemin neden seçildiğini, hangi alternatiflerin denenip neden terk edildiğini
adım adım belgelemektedir.

---

## 1. Aşama: Veri Temizleme ve Keşifsel Analiz

**Ne yapıldı:**
- İki ham dosya okunup tarih sütunları datetime formatına çevrildi.
- Eksik değerler (`Type_fuel`, `Length`) mod/medyan ile dolduruldu; `Date_lapse`
  boşluğu normal karşılandı çünkü bu alan yalnızca iptal edilen poliçelerde doludur.
- Yaş, sürüş deneyimi, araç yaşı, sözleşme süresi gibi türetilmiş değişkenler eklendi.
- Mantık dışı satırlar (negatif sürüş deneyimi, 31 satır) veri setinden çıkarıldı.
- Hasar tipi verisi (`sample_type_claim.csv`) bilinçli olarak ana tabloya satır
  satır birleştirilmedi — zaman bilgisi taşımadığı için birleştirme, hasar
  maliyetinin birden fazla yılda tekrar sayılmasına yol açardı. Bunun yerine
  ID bazında özetlenmiş, ayrı bir referans tablo (`claims_summary`) olarak tutuldu.

**Dağılım analizinden çıkan kritik bulgular:**
- 13 sayısal değişkenin tamamı istatistiksel olarak normal dağılmamaktadır
  (D'Agostino-Pearson testi, p<0.05). Bu bulgu, sonraki aşamalarda z-score
  yerine IQR, log-dönüşüm ve trend-bazlı yöntemlerin tercih edilme kararını
  doğrudan şekillendirmiştir.
- `Cost_claims_year`, `N_claims_year`, `Value_vehicle` zero-inflated yapıdadır
  (çoğunluk sıfıra yakın, az sayıda poliçede aşırı yüksek değer). Bu değişkenlerdeki
  uç değerler incelenmiş, veri hatasından çok gerçek fakat nadir olaylar
  (yüksek hasarlı poliçeler, lüks araçlar) olduğu tespit edilmiş; silinmeyip
  flag ile işaretlenerek korunmuştur.
- `Cylinder_capacity`-`Weight` (r=0.89) ve `Age`-`Driving_experience` (r=0.88)
  arasındaki yüksek korelasyon, segmentasyon aşamasında değişken seçimini
  doğrudan etkilemiştir.

**Çıktı:** `cleaned_motor_insurance_final.csv`, `claims_type_summary.csv` —
sonraki tüm aşamaların girdisi.

---

## 2. Aşama: KPI Hesaplama

**Ne yapıldı:** `calculate_kpis(data, groupby_cols)` adında tek, genel bir fonksiyon
yazıldı. Bu fonksiyon, verilen herhangi bir kırılımda (kanal, risk tipi, bölge,
dönem, veya bunların kombinasyonu) şu metrikleri hesaplar:

- **Loss ratio** (hasar maliyeti / prim) — sigortacılıkta kârlılığın temel göstergesi
- **Hasar frekansı** (poliçe başına ortalama hasar sayısı)
- **Ortalama hasar şiddeti** (hasar başına maliyet)
- **Lapse (iptal) oranı** ve yaklaşık retention oranı

Kanal bazında test edildiğinde, Broker kanalının Acente kanalına kıyasla daha
yüksek loss ratio (0.47 vs 0.40), daha yüksek hasar frekansı ve daha yüksek
lapse oranına sahip olduğu görülmüştür — bu, doğrudan aksiyona dönüştürülebilecek
bir iş içgörüsüdür.

**Çıktı:** Fonksiyon, tek başına bir dosya üretmez; 5. aşamadaki pipeline
tarafından her dönem çağrılarak dönemsel KPI raporları üretilmesini sağlar.

---

## 3. Aşama: Anomali Tespiti

Bu aşamada, birbirini tamamlayan ama farklı sorulara cevap veren **iki ayrı
anomali tespit katmanı** kurulmuştur — biri stratejik (segment/kanal seviyesi),
diğeri operasyonel (poliçe seviyesi).

**Katman 1 — Poliçe seviyesinde anomali tespiti:**
`detect_anomalies()` fonksiyonu, tek bir poliçenin (örn. hasar maliyetinin)
kendi dönemi içindeki diğer poliçelere göre ne kadar sıra dışı olduğunu ölçer
(IQR / z-score / log-zscore seçenekli). Zero-inflated `Cost_claims_year`
üzerinde `log_zscore` yöntemiyle test edildiğinde 3.293 poliçe (%3.1) anomali
olarak işaretlenmiştir — dağılıma uygun, makul bir oran. Bu katmanın çıktısı,
`build_contact_list()` fonksiyonuyla önceliklendirilmiş (Düşük/Orta/Yüksek) bir
temas listesine dönüştürülür: her dönem için ayrı bir `contact_YYYYQX.csv`
dosyası üretilir ve bu dosya, "hangi bireysel müşteriyle hemen ilgilenilmeli"
sorusuna cevap verir — doğrudan bir satış/müşteri ilişkileri ekibine
iletilebilecek somut, aksiyona dönük bir liste sunar.

**Katman 2 — Segment/kanal seviyesinde anomali tespiti:**
Burada soru farklıdır: "hangi bireysel poliçe" değil, "hangi kanal, hangi
dönemde beklenenden saptı" sorusu sorulur. Bu katman üç aşamada geliştirilmiştir:

1. İlk denemede klasik IQR yöntemi kullanıldı (kanal x çeyrek kırılımında
   loss ratio üzerinde). Sonuç: hiç anomali bulunamadı. İncelemede, loss
   ratio'nun 2015Q4'ten 2018Q4'e doğru sürekli ve düzenli azaldığı görüldü
   (muhtemelen IBNR — henüz raporlanmamış hasar etkisi — ve portföyün
   zamanla olgunlaşması). Klasik IQR, kademeli bir trendi anomali olarak
   yakalayamadığı için bu yöntem terk edildi.

2. **Trend-düzeltmeli yöntem** geliştirildi: her segment için doğrusal trend
   uydurulup, trendden sapan (residual) noktalar anomali olarak işaretlendi.
   Bu yöntemle üç net anomali bulundu (Acente 2016Q2, Broker 2016Q3, Broker 2017Q1).

3. Pipeline'a entegre edilirken bir sorun daha ortaya çıktı: expanding window
   (o ana kadarki veri) ile çalışırken, trend cari dönemin kendisini de
   içererek kuruluyordu — bu, anomalinin trend çizgisini kendine doğru çekip
   sönümlenmesine yol açıyordu (self-referential bias). Bu nedenle son sürüm,
   **`detect_forecast_anomalies`** olarak yeniden yazıldı: trend yalnızca
   cari dönemden önceki verilerle kurulur, cari dönem sadece "tahminle
   karşılaştırılan" nokta olarak kullanılır — gerçek bir ileriye dönük
   tahmin/sapma mantığı. Bu katmanın çıktısı, zaman içinde biriken (append-only)
   `anomaly_log.csv` dosyasına yazılır ve yönetime stratejik seviyede
   raporlanacak, "bu kanalda bu dönem neden sorun oldu" tarzı bir inceleme
   başlatacak sinyaller sunar.

Özetle: `contact_list` operasyonel ve anlık ("bugün kiminle konuşmalıyız"),
`anomaly_log` stratejik ve kümülatif ("zaman içinde hangi kanal tekrar tekrar
sorun çıkarıyor") sorularına hizmet eder — ikisi birbirinin yerine geçmez,
birlikte tam resmi oluşturur.

---

## 4. Aşama: Segmentasyon

**Ne yapıldı, hangi kararlar alındı:**
- Segmentasyon, poliçe-yıl değil **müşteri (ID)** seviyesinde yapıldı — aksi
  halde aynı kişi birden fazla segmente düşerdi.
- İlk denemede 7 değişkenle (Age, Power, Value_vehicle, Premium, N_claims_history,
  R_Claims_history, Seniority) K-means uygulandı; en iyi silhouette skoru
  yalnızca 0.27 çıktı (zayıf ayrım).
- Standardize edilmiş fark analizi, `Seniority` (1.79) ve `N_claims_history`
  (1.67) değişkenlerinin segmentasyonu neredeyse tek başına domine ettiğini,
  `Power` (0.015) ve `Premium` (0.009) değişkenlerinin ise pratikte hiçbir
  katkı sağlamadığını gösterdi.
- Bu nedenle segmentasyon **ikiye ayrıldı**:
  - **Risk/kıdem segmentasyonu** (Seniority, N_claims_history, R_Claims_history):
    silhouette skoru 0.577 (k=3) — güçlü bir ayrım. Üç segment ortaya çıktı:
    çoğunluk "yeni/düşük riskli" (%74.6), "eski/orta riskli sadık müşteri"
    (%16.7), ve dikkat çekici bir "yeni ama aşırı yüksek riskli" grup (%8.7,
    hasar oranı diğerlerinin 8-20 katı).
  - **Demografik/araç segmentasyonu** (Age, Power, Value_vehicle, Premium):
    silhouette skoru 0.268 (k=5) — daha zayıf ama kullanılabilir. Lüks araç
    sahipleri, yüksek performans/spor araç sahipleri, düşük güçlü araç
    sahipleri gibi anlamlı gruplar ortaya çıktı.

**Ne için kullanılıyor:** İki katmanlı yapı sayesinde hem "kime öncelik
verilmeli" (risk segmenti) hem "nasıl bir teklif sunulmalı" (demografik/araç
segmenti) sorularına ayrı ayrı cevap üretilebilmektedir.

---

## 5. Aşama: Periyodik Pipeline ve Raporlama

**Ne yapıldı:** Önceki tüm fonksiyonlar `run_periodic_pipeline()` içinde
birleştirildi. Bu fonksiyon:
1. Veriyi belirtilen periyoda (varsayılan: çeyreklik) böler.
2. Her dönem için, o ana kadarki veriyle (expanding window) KPI hesaplar.
3. Forecast-tabanlı anomali tespiti yapar ve bulunan anomalileri **append-only**
   `anomaly_log.csv` dosyasına ekler — böylece zaman içindeki tüm anomali
   geçmişi birikir, tekrar tekrar aynı analiz koşulmaz.
4. O döneme ait poliçe seviyesinde öncelikli temas listesini ayrı bir
   `contact_YYYYQX.csv` dosyasına kaydeder.
5. Dönemsel KPI raporunu ayrı dosyaya kaydeder.

Pipeline test edildiğinde, 2016Q3'te hem Acente hem Broker kanalında güçlü
anomali sinyalleri (skor 5.37 ve 5.12) yakalanmış, takip eden dönemlerde bu
etkinin kademeli olarak sönümlendiği gözlemlenmiştir — gerçek zamanlı
sistemlerde de beklenen, "şok sonrası birkaç dönem sistemin hassaslaşması"
davranışına uygun bir sonuçtur.

**Çıktı:** `reports/` klasörü altında `kpi/`, `anomalies/`, `contact_lists/`,
`segments/` alt klasörleri — her biri ayrı dosyalar halinde, tek bir zip
arşivi olarak indirilebilir durumda.

---

## Kullanılan Araçlar ve Kütüphaneler

Python, pandas, numpy, scipy (istatistiksel testler), scikit-learn (K-means,
StandardScaler, PCA, silhouette skoru), matplotlib, seaborn.

## Genel Sonuç

Bu proje; ham, gerçek dünya sigorta verisinden başlayarak, veri kalitesi
sorunlarının (zero-inflation, aşırı çarpıklık, IBNR etkisi, multicollinearity)
her birinin nasıl tespit edilip yöntemi doğrudan şekillendirdiğini gösteren,
uçtan uca ve tekrar üretilebilir bir sistem sunmaktadır. Her karar, veriden
gelen kanıtlarla desteklenmiştir — kör bir "kütüphane çağır, sonuç al"
yaklaşımı yerine, her adımda "neden bu yöntem, neden bu eşik" sorusuna
cevap aranmıştır.

---

## Örnek Çıktılar

Pipeline'ın ürettiği beş temel çıktı türü, aşağıda örnek bir dönem (2015Q4)
üzerinden açıklanmıştır.

### 1. `kpi_2015Q4.csv` — Dönemsel Performans Raporu

**Nasıl üretiliyor:** Poliçe verisi satış kanalı (Acente/Broker) bazında
gruplanarak prim ve hasar verilerinden standart sigortacılık metrikleri
hesaplanır: **loss ratio** (hasar/prim oranı), **hasar frekansı** (poliçe
başına ortalama hasar sayısı), **ortalama hasar şiddeti**, **iptal (lapse) oranı**.

**Kullanım alanı:** Kanal bazlı performansın dönemsel özeti. Hangi kanalın
daha kârlı, hangi kanalın daha riskli çalıştığını tek tabloda gösteren temel
yönetim raporudur.

### 2. `anomaly_log.csv` — Kanal Bazlı(Acente kanalı ve Broker kanalı) Erken Uyarı Kaydı

**Nasıl üretiliyor:** Her kanal için geçmiş dönemlerin loss ratio serisine
bir trend uydurulur; cari dönem bu trende hiç dahil edilmeden, yalnızca
geçmiş verilerle kurulan trendin cari dönem için öngördüğü değer ile gerçekleşen
değer karşılaştırılır. Aradaki fark istatistiksel olarak anlamlı bulunursa
(anomaly_score eşiği aşılırsa) kayıt, kalıcı bir log dosyasına eklenir.

**Kullanım alanı:** "Hangi kanal, hangi dönemde beklenen performansın dışına
çıktı" sorusunun cevabı. Log dosyası zaman içinde birikir, bu sayede tekrar
eden veya kronikleşen sorunlu dönemler geriye dönük olarak da izlenebilir —
yönetime stratejik seviyede sunulacak bir erken uyarı mekanizmasıdır.

### 3. `contact_2015Q4.csv` — Öncelikli Temas Listesi

**Nasıl üretiliyor:** Bu tablo kanal değil, **poliçe** bazlı çalışır. Hasar
maliyeti dağılımı çarpık olduğu için log dönüşümü sonrası z-score hesaplanır;
istatistiksel olarak sıra dışı poliçeler skorlarına göre üç önceliğe
(Düşük/Orta/Yüksek) ayrılır.

**Kullanım alanı:** Satış veya müşteri ilişkileri ekibine doğrudan iletilebilecek,
somut ve aksiyona dönük bir listedir — "bu dönem hangi müşteriyle öncelikli
olarak ilgilenilmeli" sorusuna cevap verir. Kanal seviyesindeki anomali
kaydından farklı olarak, bireysel poliçe seviyesinde çalışır.

### 4. `customer_risk_segments.csv` — Risk ve Sadakat Segmentasyonu

**Nasıl üretiliyor:** Müşteri kıdemi ve hasar geçmişi değişkenleri
standartlaştırılıp K-means algoritmasıyla üç segmente ayrılır. Segment sayısı
ve değişken seçimi, silhouette skoru ile optimize edilmiştir.

**Kullanım alanı:** "Bu müşteri ne kadar riskli, ne kadar sadık" sorusuna
cevap verir. Kampanya ve fiyatlandırma önceliklendirmesinde doğrudan
kullanılabilecek bir çıktıdır.

### 5. `customer_demo_segments.csv` — Demografik ve Araç Profili Segmentasyonu

**Nasıl üretiliyor:** Yaş, araç gücü, araç değeri ve prim değişkenleri
üzerinde ayrı bir K-means kümelemesi uygulanır (5 segment). Risk
segmentasyonundan bağımsız çalışan bu ikinci katman, farklı bir soruyu
cevaplamak üzere tasarlanmıştır.

**Kullanım alanı:** "Bu müşteriye nasıl bir ürün veya teklif sunulmalı"
sorusuna cevap verir — lüks segment, yüksek performans segmenti, düşük
profilli segment gibi ürün/kampanya tasarımına yön veren

In [35]:
import pandas as pd
import numpy as np


df = pd.read_csv('/content/cleaned_motor_insurance_final.csv')
df.head()

,ID,Date_start_contract,Date_last_renewal,Date_next_renewal,Date_birth,Date_driving_licence,Distribution_channel,Seniority,Policies_in_force,Max_policies,...,Vehicle_age,Contract_duration_days,Report_period_Q,flag_extreme_claim,flag_extreme_value,Cost_claims_year_capped,Value_vehicle_capped,Cost_claims_year_log,Value_vehicle_log,Premium_log
0,1,2015-11-05,2015-11-05,2016-11-05,1956-04-15,1976-03-20,0,4,1,2,...,11,0,2015Q4,False,False,0.0,7068.0,0.0,8.863474,10.010232
1,1,2015-11-05,2016-11-05,2017-11-05,1956-04-15,1976-03-20,0,4,1,2,...,12,366,2016Q4,False,False,0.0,7068.0,0.0,8.863474,9.970164
2,1,2015-11-05,2017-11-05,2018-11-05,1956-04-15,1976-03-20,0,4,2,2,...,13,731,2017Q4,False,False,0.0,7068.0,0.0,8.863474,9.975110
3,1,2015-11-05,2018-11-05,2019-11-05,1956-04-15,1976-03-20,0,4,2,2,...,14,1096,2018Q4,False,False,0.0,7068.0,0.0,8.863474,9.985068
4,2,2017-09-26,2017-09-26,2018-09-26,1956-04-15,1976-03-20,0,4,2,2,...,13,0,2017Q3,False,False,0.0,7068.0,0.0,8.863474,7.667626


In [36]:
# Genel yuvarlama yardımcı fonksiyonu
def round_numeric(data, decimals=2):
    """
    Bir DataFrame'deki tüm sayısal (float) sütunları belirtilen ondalık basamağa yuvarlar.
    ID, flag, kategori gibi sütunlara dokunmaz.
    """
    result = data.copy()
    float_cols = result.select_dtypes(include=['float64', 'float32']).columns
    result[float_cols] = result[float_cols].round(decimals)
    return result

In [37]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## KPİ Fonksiyonları

In [38]:
# Tarih sütunlarını tekrar datetime'a çevirildi
date_cols = ['Date_start_contract', 'Date_last_renewal', 'Date_next_renewal',
             'Date_birth', 'Date_driving_licence', 'Date_lapse']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print(df.shape)
df.info()

(105524, 42)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105524 entries, 0 to 105523
Data columns (total 42 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   ID                       105524 non-null  int64         
 1   Date_start_contract      105524 non-null  datetime64[ns]
 2   Date_last_renewal        105524 non-null  datetime64[ns]
 3   Date_next_renewal        105524 non-null  datetime64[ns]
 4   Date_birth               105524 non-null  datetime64[ns]
 5   Date_driving_licence     105524 non-null  datetime64[ns]
 6   Distribution_channel     105524 non-null  int64         
 7   Seniority                105524 non-null  int64         
 8   Policies_in_force        105524 non-null  int64         
 9   Max_policies             105524 non-null  int64         
 10  Max_products             105524 non-null  int64         
 11  Lapse                    105524 non-null  int64         
 12  Dat

In [39]:
# Okunabilirlik için kategori etiketleri
df['Channel_label'] = df['Distribution_channel'].map({0: 'Acente', 1: 'Broker'})
df['Risk_label'] = df['Type_risk'].map({1: 'Motorsiklet', 2: 'Van', 3: 'Binek araç', 4: 'Tarım aracı'})
df['Area_label'] = df['Area'].map({0: 'Kırsal', 1: 'Şehir'})

In [40]:
# Genel KPI hesaplama fonksiyonu
def calculate_kpis(data, groupby_cols):
    """
    Verilen kırılım(lar)a göre temel sigorta KPI'larını hesaplar.
    groupby_cols: liste, örn. ['Channel_label'] veya ['Channel_label', 'Risk_label']
    """
    grouped = data.groupby(groupby_cols).agg(
        n_policies=('ID', 'count'),
        n_unique_customers=('ID', 'nunique'),
        total_premium=('Premium', 'sum'),
        total_claim_cost=('Cost_claims_year', 'sum'),
        total_claims=('N_claims_year', 'sum'),
        n_lapsed=('Lapse', 'sum'),
        n_extreme_claim=('flag_extreme_claim', 'sum')
    ).reset_index()

    # Loss ratio: hasar maliyeti / prim
    grouped['loss_ratio'] = grouped['total_claim_cost'] / grouped['total_premium']

    # Hasar frekansı: poliçe başına ortalama hasar sayısı
    grouped['claim_frequency'] = grouped['total_claims'] / grouped['n_policies']

    # Ortalama hasar şiddeti: hasar başına maliyet (0'a bölme riski var, kontrol ediyoruz)
    grouped['avg_claim_severity'] = np.where(
        grouped['total_claims'] > 0,
        grouped['total_claim_cost'] / grouped['total_claims'],
        0
    )

    # Lapse (iptal) oranı
    grouped['lapse_rate'] = grouped['n_lapsed'] / grouped['n_policies']

    # Basitleştirilmiş retention oranı (not: gerçek retention, aynı müşterinin
    # bir sonraki döneme geçip geçmediğini takip eder; burada yaklaşık gösterge)
    grouped['retention_rate_approx'] = 1 - grouped['lapse_rate']

    return grouped.sort_values('n_policies', ascending=False)

In [41]:
# Kanal bazında KPI'lar (ilanın "satış kanalı performansı" maddesi)
kpi_by_channel = calculate_kpis(df, ['Channel_label'])
kpi_by_channel = round_numeric(calculate_kpis(df, ['Channel_label']))
kpi_by_channel

,Channel_label,n_policies,n_unique_customers,total_premium,total_claim_cost,total_claims,n_lapsed,n_extreme_claim,loss_ratio,claim_frequency,avg_claim_severity,lapse_rate,retention_rate_approx
0,Acente,57897,28082,1594911347,630081025,21189,11614,49,0.40,0.37,29736.23,0.20,0.80
1,Broker,47627,25413,1428203399,672249242,20457,11799,57,0.47,0.43,32861.58,0.25,0.75


In [42]:
# Risk tipi bazında KPI'lar
kpi_by_risk = calculate_kpis(df, ['Risk_label'])
kpi_by_channel = round_numeric(calculate_kpis(df, ['Channel_label']))
kpi_by_risk

,Risk_label,n_policies,n_unique_customers,total_premium,total_claim_cost,total_claims,n_lapsed,n_extreme_claim,loss_ratio,claim_frequency,avg_claim_severity,lapse_rate,retention_rate_approx
0,Binek araç,82974,42242,2511771629,1108337207,32914,18185,93,0.441257,0.396678,33673.731755,0.219165,0.780835
3,Van,13211,6602,394063929,168904335,7499,3037,11,0.428622,0.567633,22523.581144,0.229884,0.770116
1,Motorsiklet,8488,4293,111099613,25069458,1232,2023,2,0.225648,0.145146,20348.586039,0.238336,0.761664
2,Tarım aracı,851,358,6179575,19267,1,168,0,0.003118,0.001175,19267.000000,0.197415,0.802585


In [43]:
# Kanal x Risk tipi çapraz kırılım
kpi_by_channel_risk = calculate_kpis(df, ['Channel_label', 'Risk_label'])
kpi_by_channel = round_numeric(calculate_kpis(df, ['Channel_label']))
kpi_by_channel_risk

,Channel_label,Risk_label,n_policies,n_unique_customers,total_premium,total_claim_cost,total_claims,n_lapsed,n_extreme_claim,loss_ratio,claim_frequency,avg_claim_severity,lapse_rate,retention_rate_approx
0,Acente,Binek araç,45647,22136,1340813633,538262084,16955,8892,42,0.401444,0.371437,31746.510410,0.194799,0.805201
4,Broker,Binek araç,37327,20106,1170957996,570075123,15959,9293,51,0.486845,0.427546,35721.230842,0.248962,0.751038
7,Broker,Van,6738,3459,210704984,92033881,4011,1676,6,0.436790,0.595280,22945.370481,0.248738,0.751262
3,Acente,Van,6473,3143,183358945,76870454,3488,1361,5,0.419235,0.538854,22038.547592,0.210258,0.789742
1,Acente,Motorsiklet,5080,2518,65718735,14948487,746,1217,2,0.227462,0.146850,20038.186327,0.239567,0.760433
5,Broker,Motorsiklet,3408,1775,45380878,10120971,486,806,0,0.223023,0.142606,20825.043210,0.236502,0.763498
2,Acente,Tarım aracı,697,285,5020034,0,0,144,0,0.000000,0.000000,0.000000,0.206600,0.793400
6,Broker,Tarım aracı,154,73,1159541,19267,1,24,0,0.016616,0.006494,19267.000000,0.155844,0.844156


In [44]:
# Bölge (Area) bazında KPI'lar
kpi_by_area = calculate_kpis(df, ['Area_label'])
kpi_by_channel = round_numeric(calculate_kpis(df, ['Channel_label']))
kpi_by_area

,Area_label,n_policies,n_unique_customers,total_premium,total_claim_cost,total_claims,n_lapsed,n_extreme_claim,loss_ratio,claim_frequency,avg_claim_severity,lapse_rate,retention_rate_approx
0,Kırsal,76635,38894,2159988164,896199271,28553,16289,78,0.414909,0.372584,31387.219241,0.212553,0.787447
1,Şehir,28889,14601,863126582,406130996,13093,7124,28,0.470535,0.453217,31018.941114,0.246599,0.753401


In [45]:
# Zaman bazlı (dönemsel) KPI trend tablosu
# Not: Report_period_Q, 1. aşamada oluşturulmuştu; CSV'den geldiği için string'e dönmüş olabilir,
# tekrar Period tipine çeviriyoruz
df['Report_period_Q'] = pd.to_datetime(df['Date_last_renewal']).dt.to_period('Q')

kpi_trend = calculate_kpis(df, ['Report_period_Q'])
kpi_trend = kpi_trend.sort_values('Report_period_Q')
kpi_trend

,Report_period_Q,n_policies,n_unique_customers,total_premium,total_claim_cost,total_claims,n_lapsed,n_extreme_claim,loss_ratio,claim_frequency,avg_claim_severity,lapse_rate,retention_rate_approx
0,2015Q4,4555,4555,130835608,96699271,3239,1250,6,0.739090,0.711087,29854.668416,0.274424,0.725576
1,2016Q1,7388,7388,212718390,160937551,5407,1938,15,0.756576,0.731862,29764.666358,0.262317,0.737683
2,2016Q2,8213,8213,232516639,176159022,5680,2268,13,0.757619,0.691587,31013.912324,0.276148,0.723852
3,2016Q3,8308,8308,239782611,184654465,5643,2097,13,0.770091,0.679225,32722.747652,0.252407,0.747593
4,2016Q4,7510,7510,220177292,114283482,4128,1877,9,0.519052,0.549667,27684.952035,0.249933,0.750067
5,2017Q1,7510,7510,217954089,109936378,3091,1940,9,0.504402,0.411585,35566.605629,0.258322,0.741678
6,2017Q2,8514,8514,241904403,95490736,3104,2190,11,0.394746,0.364576,30763.768041,0.257223,0.742777
7,2017Q3,8919,8919,258537586,92909690,3086,2159,7,0.359366,0.346003,30106.834089,0.242067,0.757933
8,2017Q4,8797,8797,255479463,97041569,3114,1901,7,0.379841,0.353984,31162.995825,0.216096,0.783904
9,2018Q1,8690,8690,249359278,82571384,2509,1871,8,0.331134,0.288723,32910.077322,0.215305,0.784695


In [46]:
# Kanal bazında zaman içinde trend (3. aşamada anomali tespiti bunu kullanacak)
kpi_channel_trend = calculate_kpis(df, ['Report_period_Q', 'Channel_label'])
kpi_channel_trend = kpi_channel_trend.sort_values(['Channel_label', 'Report_period_Q'])
kpi_channel_trend.head(20)

,Report_period_Q,Channel_label,n_policies,n_unique_customers,total_premium,total_claim_cost,total_claims,n_lapsed,n_extreme_claim,loss_ratio,claim_frequency,avg_claim_severity,lapse_rate,retention_rate_approx
0,2015Q4,Acente,2396,2396,67196515,46542727,1638,563,3,0.692636,0.683639,28414.363248,0.234975,0.765025
2,2016Q1,Acente,3870,3870,106411179,69203451,2665,920,5,0.650340,0.688630,25967.523827,0.237726,0.762274
4,2016Q2,Acente,4376,4376,118694239,96301206,2829,1061,6,0.811339,0.646481,34040.723224,0.242459,0.757541
6,2016Q3,Acente,4575,4575,126394568,73243963,2987,1028,3,0.579487,0.652896,24520.911617,0.224699,0.775301
8,2016Q4,Acente,4234,4234,119058891,51768590,2166,936,2,0.434815,0.511573,23900.549400,0.221068,0.778932
10,2017Q1,Acente,4084,4084,113052454,68469442,1587,956,6,0.605643,0.388590,43143.945810,0.234084,0.765916
12,2017Q2,Acente,4669,4669,126884070,44165741,1520,1111,5,0.348079,0.325552,29056.408553,0.237952,0.762048
14,2017Q3,Acente,4948,4948,137559570,42276419,1563,1076,5,0.307332,0.315885,27048.252719,0.217462,0.782538
16,2017Q4,Acente,4938,4938,138648596,48687886,1588,1014,3,0.351160,0.321588,30659.877834,0.205346,0.794654
18,2018Q1,Acente,4768,4768,131355755,38872572,1284,947,4,0.295934,0.269295,30274.588785,0.198616,0.801384


## Anomali Tespit Fonksiyonu

In [47]:
#  Gerekli kütüphaneler
from scipy import stats

In [48]:
# Genel anomali tespit fonksiyonu (tek bir metrik serisi üzerinde çalışır)
def detect_anomalies(data, value_col, method='iqr', threshold=1.5):
    """
    Bir metrik sütunundaki anomalileri tespit eder.
    method: 'iqr', 'zscore', veya 'log_zscore'
    threshold: iqr için çarpan (varsayılan 1.5), zscore için sigma sayısı (örn. 2.5)
    Döndürür: orijinal veriye eklenmiş anomaly_flag ve anomaly_score sütunları
    """
    result = data.copy()
    series = result[value_col]

    if method == 'iqr':
        Q1 = series.quantile(0.25)
        Q3 = series.quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - threshold * IQR
        upper = Q3 + threshold * IQR
        result['anomaly_flag'] = (series < lower) | (series > upper)
        # şiddet skoru: sınırdan ne kadar uzakta (normalize)
        result['anomaly_score'] = np.where(
            series > upper, (series - upper) / (IQR + 1e-9),
            np.where(series < lower, (lower - series) / (IQR + 1e-9), 0)
        )

    elif method == 'zscore':
        mean = series.mean()
        std = series.std()
        z = (series - mean) / (std + 1e-9)
        result['anomaly_flag'] = z.abs() > threshold
        result['anomaly_score'] = z.abs()

    elif method == 'log_zscore':
        # zero-inflated / sağa çarpık değişkenler için (Cost_claims_year gibi)
        log_series = np.log1p(series)
        mean = log_series.mean()
        std = log_series.std()
        z = (log_series - mean) / (std + 1e-9)
        result['anomaly_flag'] = z.abs() > threshold
        result['anomaly_score'] = z.abs()

    else:
        raise ValueError("method 'iqr', 'zscore' veya 'log_zscore' olmalı")

    return result

In [49]:
# Fonksiyonu tek bir metrikte test etme (Cost_claims_year, zero-inflated olduğu için log_zscore)
test_result = detect_anomalies(df, 'Cost_claims_year', method='log_zscore', threshold=2.5)

print("Toplam anomali sayısı:", test_result['anomaly_flag'].sum())
test_result[test_result['anomaly_flag']][['ID', 'Channel_label', 'Risk_label',
                                            'Cost_claims_year', 'anomaly_score']].sort_values(
                                            'anomaly_score', ascending=False).head(10)

Toplam anomali sayısı: 3293


,ID,Channel_label,Risk_label,Cost_claims_year,anomaly_score
55141,27326,Broker,Binek araç,26085324,3.992849
63262,31466,Acente,Binek araç,23628518,3.967003
95787,48099,Acente,Binek araç,12880973,3.808476
92937,46596,Acente,Binek araç,7196887,3.656378
44035,21404,Acente,Binek araç,5965665,3.607352
60853,30233,Acente,Binek araç,5010194,3.561745
69093,34467,Broker,Binek araç,4854677,3.553506
48895,24000,Broker,Binek araç,4065848,3.507174
55187,27351,Acente,Van,3746928,3.485830
74368,37246,Broker,Binek araç,3637581,3.478091


In [50]:
# Öncelikli temas listesi çıktısı (ATS bildirim simülasyonu)
def build_contact_list(anomaly_data, id_col='ID', extra_cols=None):
    """
    Anomali flag'i true olan satırları, ATS'e bildirilecek formatta düzenler.
    """
    cols = [id_col, 'anomaly_score', 'anomaly_flag']
    if extra_cols:
        cols = [id_col] + extra_cols + ['anomaly_score', 'anomaly_flag']

    contact_list = anomaly_data[anomaly_data['anomaly_flag']][cols].copy()
    contact_list = contact_list.sort_values('anomaly_score', ascending=False)
    contact_list['priority'] = pd.qcut(contact_list['anomaly_score'],
                                         q=min(3, contact_list['anomaly_score'].nunique()),
                                         labels=['Düşük', 'Orta', 'Yüksek'], duplicates='drop')
    return contact_list

In [51]:
# Öncelikli temas listesini test etme
contact_list = round_numeric(build_contact_list(test_result, id_col='ID',
                                    extra_cols=['Channel_label', 'Risk_label', 'Cost_claims_year']))
contact_list.head(15)

,ID,Channel_label,Risk_label,Cost_claims_year,anomaly_score,anomaly_flag,priority
55141,27326,Broker,Binek araç,26085324,3.99,True,Yüksek
63262,31466,Acente,Binek araç,23628518,3.97,True,Yüksek
95787,48099,Acente,Binek araç,12880973,3.81,True,Yüksek
92937,46596,Acente,Binek araç,7196887,3.66,True,Yüksek
44035,21404,Acente,Binek araç,5965665,3.61,True,Yüksek
60853,30233,Acente,Binek araç,5010194,3.56,True,Yüksek
69093,34467,Broker,Binek araç,4854677,3.55,True,Yüksek
48895,24000,Broker,Binek araç,4065848,3.51,True,Yüksek
55187,27351,Acente,Van,3746928,3.49,True,Yüksek
74368,37246,Broker,Binek araç,3637581,3.48,True,Yüksek


In [52]:
#  priority dağılımını kontrol etme
print(contact_list['priority'].value_counts())
print("Toplam satır:", len(contact_list))

priority
Düşük     1098
Yüksek    1098
Orta      1097
Name: count, dtype: int64
Toplam satır: 3293


In [53]:
# Trend-düzeltmeli anomali tespiti
# Mantık: her kanal için zaman içindeki doğrusal trendi hesapla, trendden sapan (residual)
# noktaları anomali olarak işaretle. Böylece "genel azalış" normal kabul edilir,
# sadece trendin DIŞINA çıkan ani sıçramalar/düşüşler yakalanır.

def detect_trend_anomalies(kpi_data, metric_col, time_col, group_col, threshold=1.5):
    results = []
    for group, sub in kpi_data.groupby(group_col):
        sub = sub.sort_values(time_col).reset_index(drop=True)
        x = np.arange(len(sub))
        y = sub[metric_col].values

        if len(x) < 4:  # trend hesaplamak için minimum nokta gerekli
            sub['anomaly_flag'] = False
            sub['anomaly_score'] = 0
            results.append(sub)
            continue

        # Doğrusal trend uydur
        coeffs = np.polyfit(x, y, deg=1)
        trend = np.polyval(coeffs, x)
        residuals = y - trend

        # Residual'lerin kendi std'sine göre anomali skoru
        resid_std = residuals.std() + 1e-9
        z_resid = residuals / resid_std

        sub['trend_value'] = trend
        sub['residual'] = residuals
        sub['anomaly_score'] = np.abs(z_resid)
        sub['anomaly_flag'] = np.abs(z_resid) > threshold

        results.append(sub)

    return pd.concat(results).reset_index(drop=True)

In [54]:
# Test etme
trend_anomaly = detect_trend_anomalies(sample_kpi_mature, 'loss_ratio',
                                         'Report_period_Q', 'Channel_label', threshold=1.5)

print(trend_anomaly[['Report_period_Q', 'Channel_label', 'loss_ratio',
                       'trend_value', 'residual', 'anomaly_score', 'anomaly_flag']]
      .sort_values(['Channel_label', 'Report_period_Q']))

   Report_period_Q Channel_label  loss_ratio  trend_value  residual  \
0           2015Q4        Acente    0.451173     0.530708 -0.079534   
1           2016Q1        Acente    0.558283     0.498420  0.059863   
2           2016Q2        Acente    0.491458     0.466132  0.025326   
3           2016Q3        Acente    0.469507     0.433844  0.035664   
4           2016Q4        Acente    0.357978     0.401556 -0.043578   
5           2017Q1        Acente    0.411048     0.369268  0.041780   
6           2017Q2        Acente    0.340993     0.336980  0.004013   
7           2017Q3        Acente    0.247641     0.304692 -0.057050   
8           2017Q4        Acente    0.293590     0.272404  0.021186   
9           2018Q1        Acente    0.209735     0.240116 -0.030381   
10          2018Q2        Acente    0.230539     0.207828  0.022712   
11          2015Q4        Broker    0.808948     0.774812  0.034136   
12          2016Q1        Broker    0.907914     0.714154  0.193760   
13    

## 3. Müşteri/Poliçe Segmentasyonu

In [65]:
# HÜCRE 20: Kütüphaneler
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

In [66]:
# HÜCRE 21: Segmentasyon için değişken seçimi
# Not: 1. aşamada bulduğumuz yüksek korelasyonlar nedeniyle (Cylinder_capacity-Weight: 0.89,
# Age-Driving_experience: 0.88) her ikisini birden almıyoruz, birini temsilci olarak seçiyoruz.

segmentation_features = [
    'Age',              # Driving_experience yerine (0.88 korelasyonlu, biri yeterli)
    'Power',            # Cylinder_capacity/Weight yerine (araç büyüklüğünü temsil eder)
    'Value_vehicle',
    'Premium',
    'N_claims_history',
    'R_Claims_history',
    'Seniority',
]

print("Seçilen değişkenler:", segmentation_features)
print("Eksik değer kontrolü:")
print(df[segmentation_features].isna().sum())

Seçilen değişkenler: ['Age', 'Power', 'Value_vehicle', 'Premium', 'N_claims_history', 'R_Claims_history', 'Seniority']
Eksik değer kontrolü:
Age                 0
Power               0
Value_vehicle       0
Premium             0
N_claims_history    0
R_Claims_history    0
Seniority           0
dtype: int64


In [67]:
claims_summary = pd.read_csv('claims_type_summary.csv')
claims_summary.head()

,ID,total_claim_cost_by_type,n_claim_records,dominant_claim_type
0,28,33202,1,complaint
1,36,5735,1,travel assistance
2,42,575,1,travel assistance
3,60,13890,2,travel assistance
4,77,157493,1,complaint


In [68]:
# HÜCRE 22: ID bazında özet tablo oluşturma
# Not: segmentasyon poliçe-yıl satırında değil, MÜŞTERİ (ID) seviyesinde yapılmalı,
# çünkü aynı kişi birden fazla satırda tekrar ediyor. Her ID için en güncel/ortalama değerleri alıyoruz.

customer_profile = df.groupby('ID').agg(
    Age=('Age', 'max'),                          # en güncel yaş
    Power=('Power', 'mean'),
    Value_vehicle=('Value_vehicle', 'mean'),
    Premium=('Premium', 'mean'),
    N_claims_history=('N_claims_history', 'max'),
    R_Claims_history=('R_Claims_history', 'max'),
    Seniority=('Seniority', 'max'),
    Channel_label=('Channel_label', 'first'),
    Risk_label=('Risk_label', 'first'),
).reset_index()

# claims_summary (1. aşamadan) ile birleştirme - hasar tipi bilgisi
customer_profile = customer_profile.merge(claims_summary, on='ID', how='left')
customer_profile['n_claim_records'] = customer_profile['n_claim_records'].fillna(0)
customer_profile['total_claim_cost_by_type'] = customer_profile['total_claim_cost_by_type'].fillna(0)

print(customer_profile.shape)
customer_profile.head()

(53495, 13)


,ID,Age,Power,Value_vehicle,Premium,N_claims_history,R_Claims_history,Seniority,Channel_label,Risk_label,total_claim_cost_by_type,n_claim_records,dominant_claim_type
0,1,62,80.0,7068.0,21703.25,0,0,4,Acente,Motorsiklet,0.0,0.0,NaN
1,2,62,80.0,7068.0,11860.00,0,0,4,Acente,Motorsiklet,0.0,0.0,NaN
2,3,43,85.0,16030.0,12804.75,0,0,15,Acente,Binek araç,0.0,0.0,NaN
3,4,45,6.0,126182.0,11779.00,0,0,3,Acente,Motorsiklet,0.0,0.0,NaN
4,5,44,6.0,3000.0,8065.00,0,0,3,Acente,Motorsiklet,0.0,0.0,NaN


In [69]:
# HÜCRE 23: Segmentasyon fonksiyonu
def segment_customers(data, features, n_clusters=None, max_k=8, random_state=42):
    """
    K-means ile müşteri segmentasyonu yapar.
    n_clusters=None ise silhouette skoruna göre en iyi k'yı otomatik seçer.
    Döndürür: (segmented_data, model, scaler, best_k, silhouette_scores)
    """
    X = data[features].copy()

    # Ölçeklendirme (K-means mesafe bazlı olduğu için şart)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    silhouette_scores = {}

    if n_clusters is None:
        # Elbow yerine silhouette skoru ile en iyi k'yı bul (2'den max_k'ya kadar)
        for k in range(2, max_k + 1):
            km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
            labels = km.fit_predict(X_scaled)
            score = silhouette_score(X_scaled, labels)
            silhouette_scores[k] = score

        best_k = max(silhouette_scores, key=silhouette_scores.get)
    else:
        best_k = n_clusters

    # Final model
    final_model = KMeans(n_clusters=best_k, random_state=random_state, n_init=10)
    labels = final_model.fit_predict(X_scaled)

    result = data.copy()
    result['segment'] = labels

    return result, final_model, scaler, best_k, silhouette_scores

In [76]:
# HÜCRE 25-fonksiyon: Segment profil özeti fonksiyonu (tekrar eklendi)
def summarize_segments(segmented_data, features, segment_col='segment'):
    profile = segmented_data.groupby(segment_col).agg(
        n_customers=('ID', 'count'),
        **{f'avg_{f}': (f, 'mean') for f in features}
    ).reset_index()

    profile['pct_of_total'] = (profile['n_customers'] / profile['n_customers'].sum() * 100).round(1)

    return profile

In [74]:
# HÜCRE 25b: Segmentasyon değişkenleri arası korelasyon (yeni bir gizli multicollinearity var mı?)
customer_profile[segmentation_features].corr()

,Age,Power,Value_vehicle,Premium,N_claims_history,R_Claims_history,Seniority
Age,1.000000,-0.046545,0.042920,-0.109457,0.054691,-0.044325,0.194124
Power,-0.046545,1.000000,0.169087,0.357866,0.063404,0.068650,-0.061778
Value_vehicle,0.042920,0.169087,1.000000,0.096557,0.036579,0.019032,0.015809
Premium,-0.109457,0.357866,0.096557,1.000000,0.080989,0.089968,-0.057717
N_claims_history,0.054691,0.063404,0.036579,0.080989,1.000000,0.287619,0.451107
R_Claims_history,-0.044325,0.068650,0.019032,0.089968,0.287619,1.000000,-0.035035
Seniority,0.194124,-0.061778,0.015809,-0.057717,0.451107,-0.035035,1.000000


In [79]:
# HÜCRE 25d: Risk/kıdem odaklı segmentasyon (baskın değişkenlerle, amaçlı)
risk_features = ['Seniority', 'N_claims_history', 'R_Claims_history']

segmented_risk, model_risk, scaler_risk, best_k_risk, sil_risk = segment_customers(
    customer_profile, risk_features, n_clusters=None, max_k=6
)

print("Risk segmentasyonu silhouette skorları:")
for k, s in sil_risk.items():
    print(f"  k={k}: {s:.4f}")
print(f"Seçilen k: {best_k_risk}")

summarize_segments(segmented_risk, risk_features).pipe(round_numeric)

Risk segmentasyonu silhouette skorları:
  k=2: 0.5387
  k=3: 0.5770
  k=4: 0.5607
  k=5: 0.4614
  k=6: 0.4714
Seçilen k: 3


,segment,n_customers,avg_Seniority,avg_N_claims_history,avg_R_Claims_history,pct_of_total
0,0,8948,15.90,7.58,27.84,16.7
1,1,39892,3.80,1.24,10.44,74.6
2,2,4655,3.42,4.77,216.01,8.7


In [80]:
# HÜCRE 25e: Demografik/araç odaklı segmentasyon (Power, Premium'a asıl alan açılıyor)
demo_features = ['Age', 'Power', 'Value_vehicle', 'Premium']

segmented_demo, model_demo, scaler_demo, best_k_demo, sil_demo = segment_customers(
    customer_profile, demo_features, n_clusters=None, max_k=6
)

print("Demografik segmentasyon silhouette skorları:")
for k, s in sil_demo.items():
    print(f"  k={k}: {s:.4f}")
print(f"Seçilen k: {best_k_demo}")

summarize_segments(segmented_demo, demo_features)

Demografik segmentasyon silhouette skorları:
  k=2: 0.1973
  k=3: 0.2423
  k=4: 0.2400
  k=5: 0.2682
  k=6: 0.2471
Seçilen k: 5


,segment,n_customers,avg_Age,avg_Power,avg_Value_vehicle,avg_Premium,pct_of_total
0,0,7932,46.829047,107.234367,2.278235e+06,31361.100237,14.8
1,1,5673,42.927375,124.954521,1.346634e+05,54764.457092,10.6
2,2,4812,47.882585,21.274730,1.251745e+05,11465.402154,9.0
3,3,19064,36.967530,96.917908,1.300871e+05,26336.716604,35.6
4,4,16014,59.470338,92.548707,2.114639e+05,26161.508154,29.9


## 4. Periyodik Pipeline ve Raporlama

In [81]:
# HÜCRE 29: Klasör yapısı oluşturma
import os
from datetime import datetime

REPORT_DIR = 'reports'
os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(f'{REPORT_DIR}/kpi', exist_ok=True)
os.makedirs(f'{REPORT_DIR}/anomalies', exist_ok=True)
os.makedirs(f'{REPORT_DIR}/contact_lists', exist_ok=True)
os.makedirs(f'{REPORT_DIR}/segments', exist_ok=True)

print("Klasör yapısı hazır:", os.listdir(REPORT_DIR))

Klasör yapısı hazır: ['anomalies', 'kpi', 'contact_lists', 'segments']


In [82]:
# Önceki (yanlış) çalıştırmadan kalan dosyaları temizleme
# Çünkü eski anomaly_log.csv boştu ama pipeline yine de klasörleri oluşturmuştu
import shutil
if os.path.exists(REPORT_DIR):
    shutil.rmtree(REPORT_DIR)

os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(f'{REPORT_DIR}/kpi', exist_ok=True)
os.makedirs(f'{REPORT_DIR}/anomalies', exist_ok=True)
os.makedirs(f'{REPORT_DIR}/contact_lists', exist_ok=True)
os.makedirs(f'{REPORT_DIR}/segments', exist_ok=True)

In [83]:
# (GÜNCELLENMİŞ): Ana pipeline fonksiyonu
def run_periodic_pipeline(data, period='Q', groupby_col='Channel_label',
                            anomaly_metric='loss_ratio', anomaly_threshold=1.5,
                            min_history=3):
    """
    Veriyi belirtilen periyotlara böler, her dönem için:
    - KPI hesaplar
    - Forecast tabanlı (sadece geçmişle kurulan trend) anomali tespiti yapar
    - Poliçe seviyesinde öncelikli temas listesi çıkarır
    - Sonuçları tarihli dosyalar olarak kaydeder
    - Anomali log dosyasına ekler (append)

    period: 'M' (aylık) veya 'Q' (çeyreklik)
    """
    data = data.copy()
    period_col = f'Report_period_{period}'
    data[period_col] = pd.to_datetime(data['Date_last_renewal']).dt.to_period(period)

    periods = sorted(data[period_col].dropna().unique())
    run_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    anomaly_log_path = f'{REPORT_DIR}/anomalies/anomaly_log.csv'
    all_period_kpis = []

    for current_period in periods:
        cumulative_data = data[data[period_col] <= current_period]

        period_kpi = calculate_kpis(cumulative_data[cumulative_data[period_col] == current_period],
                                      [groupby_col])
        period_kpi[period_col] = str(current_period)
        all_period_kpis.append(period_kpi)

        history_kpi = calculate_kpis(cumulative_data, [period_col, groupby_col])
        history_kpi[period_col] = history_kpi[period_col].astype(str)

        forecast_result = detect_forecast_anomalies(history_kpi, anomaly_metric,
                                                        period_col, groupby_col,
                                                        threshold=anomaly_threshold,
                                                        min_history=min_history)
        current_anomalies = forecast_result[forecast_result[period_col] == str(current_period)]
        current_anomalies = current_anomalies[current_anomalies['anomaly_flag']]

        if len(current_anomalies) > 0:
            current_anomalies = current_anomalies.copy()
            current_anomalies['run_timestamp'] = run_timestamp
            header = not os.path.exists(anomaly_log_path)
            current_anomalies = round_numeric(current_anomalies)
            current_anomalies.to_csv(anomaly_log_path, mode='a', header=header, index=False)

        period_policies = data[data[period_col] == current_period]
        if len(period_policies) > 0:
            policy_anomaly = detect_anomalies(period_policies, 'Cost_claims_year',
                                                method='log_zscore', threshold=2.5)
            contact_list = build_contact_list(policy_anomaly, id_col='ID',
                                                 extra_cols=[groupby_col, 'Cost_claims_year'])
            if len(contact_list) > 0:
                contact_list = round_numeric(contact_list)
                contact_list.to_csv(f'{REPORT_DIR}/contact_lists/contact_{current_period}.csv',
                                      index=False)
        period_kpi = round_numeric(period_kpi)
        period_kpi.to_csv(f'{REPORT_DIR}/kpi/kpi_{current_period}.csv', index=False)

    full_kpi_history = pd.concat(all_period_kpis, ignore_index=True)
    full_kpi_history.to_csv(f'{REPORT_DIR}/kpi/kpi_all_periods.csv', index=False)

    return full_kpi_history

In [84]:
# Forecast tabanlı anomali fonksiyonu (detect_trend_anomalies'in yerini alacak)
def detect_forecast_anomalies(kpi_data, metric_col, time_col, group_col,
                                 threshold=1.5, min_history=3):
    """
    Her nokta için trend SADECE ondan önceki verilerle kurulur (cari nokta trende dahil edilmez).
    Böylece 'trend kendine çekiliyor' problemi ortadan kalkar - gerçek bir ileriye dönük
    tahmin/sapma mantığı çalışır.
    """
    results = []
    for group, sub in kpi_data.groupby(group_col):
        sub = sub.sort_values(time_col).reset_index(drop=True)
        n = len(sub)

        sub['trend_value'] = np.nan
        sub['residual'] = np.nan
        sub['anomaly_score'] = 0.0
        sub['anomaly_flag'] = False

        for i in range(min_history, n):
            hist_x = np.arange(i)
            hist_y = sub[metric_col].values[:i]

            coeffs = np.polyfit(hist_x, hist_y, deg=1)
            pred = np.polyval(coeffs, i)

            resid_hist = hist_y - np.polyval(coeffs, hist_x)
            resid_std = resid_hist.std() + 1e-9

            actual = sub[metric_col].values[i]
            resid = actual - pred
            z = resid / resid_std

            sub.loc[i, 'trend_value'] = pred
            sub.loc[i, 'residual'] = resid
            sub.loc[i, 'anomaly_score'] = abs(z)
            sub.loc[i, 'anomaly_flag'] = abs(z) > threshold

        results.append(sub)

    return pd.concat(results).reset_index(drop=True)

In [85]:
# Pipeline'ı çalıştırma (çeyreklik)
pipeline_result = run_periodic_pipeline(df, period='Q', groupby_col='Channel_label',
                                          anomaly_metric='loss_ratio', anomaly_threshold=1.5)

print("Pipeline tamamlandı.")
print("Toplam dönem-kanal satırı:", len(pipeline_result))
pipeline_result.head(10)

Pipeline tamamlandı.
Toplam dönem-kanal satırı: 26


,Channel_label,n_policies,n_unique_customers,total_premium,total_claim_cost,total_claims,n_lapsed,n_extreme_claim,loss_ratio,claim_frequency,avg_claim_severity,lapse_rate,retention_rate_approx,Report_period_Q
0,Acente,2396,2396,67196515,46542727,1638,563,3,0.692636,0.683639,28414.363248,0.234975,0.765025,2015Q4
1,Broker,2159,2159,63639093,50156544,1601,687,3,0.788140,0.741547,31328.259838,0.318203,0.681797,2015Q4
2,Acente,3870,3870,106411179,69203451,2665,920,5,0.650340,0.688630,25967.523827,0.237726,0.762274,2016Q1
3,Broker,3518,3518,106307211,91734100,2742,1018,10,0.862915,0.779420,33455.178702,0.289369,0.710631,2016Q1
4,Acente,4376,4376,118694239,96301206,2829,1061,6,0.811339,0.646481,34040.723224,0.242459,0.757541,2016Q2
5,Broker,3837,3837,113822400,79857816,2851,1207,7,0.701600,0.743028,28010.458085,0.314569,0.685431,2016Q2
6,Acente,4575,4575,126394568,73243963,2987,1028,3,0.579487,0.652896,24520.911617,0.224699,0.775301,2016Q3
7,Broker,3733,3733,113388043,111410502,2656,1069,10,0.982560,0.711492,41946.725151,0.286365,0.713635,2016Q3
8,Acente,4234,4234,119058891,51768590,2166,936,2,0.434815,0.511573,23900.549400,0.221068,0.778932,2016Q4
9,Broker,3276,3276,101118401,62514892,1962,941,7,0.618235,0.598901,31862.839959,0.287241,0.712759,2016Q4


In [86]:
# Anomali log'unu kontrol etme
if os.path.exists(f'{REPORT_DIR}/anomalies/anomaly_log.csv'):
    anomaly_log = pd.read_csv(f'{REPORT_DIR}/anomalies/anomaly_log.csv')
    print(f"Toplam {len(anomaly_log)} anomali kaydı bulundu:")
    print(anomaly_log[['Report_period_Q', 'Channel_label', 'loss_ratio',
                         'anomaly_score']].to_string(index=False))
else:
    print("Henüz anomali log dosyası oluşmadı.")

Toplam 7 anomali kaydı bulundu:
Report_period_Q Channel_label  loss_ratio  anomaly_score
         2016Q3        Acente        0.58           5.37
         2016Q3        Broker        0.98           5.12
         2016Q4        Acente        0.43           2.49
         2016Q4        Broker        0.62           3.50
         2017Q1        Acente        0.61           1.59
         2017Q1        Broker        0.40           2.69
         2017Q2        Acente        0.35           1.58


In [87]:
# Segment anlık görüntüsünü (snapshot) ayrıca kaydetme
# Not: segmentasyon periyodik değil, tüm müşteri tabanı üzerinde tek seferlik çalıştırılır,
# ama her pipeline çalışmasında güncel görüntüsünü rapor klasörüne de ekliyoruz.
round_numeric(segmented_risk).to_csv(f'{REPORT_DIR}/segments/customer_risk_segments.csv', index=False)
round_numeric(segmented_demo).to_csv(f'{REPORT_DIR}/segments/customer_demo_segments.csv', index=False)

print("Segment raporları kaydedildi.")

Segment raporları kaydedildi.


In [88]:
# Tüm rapor klasörünü zip'leyip indirme
import shutil

shutil.make_archive('insurance_reports', 'zip', REPORT_DIR)

from google.colab import files
files.download('insurance_reports.zip')

print("Rapor klasörü yapısı:")
for root, dirs, filenames in os.walk(REPORT_DIR):
    for f in filenames:
        print(os.path.join(root, f))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Rapor klasörü yapısı:
reports/anomalies/anomaly_log.csv
reports/kpi/kpi_2015Q4.csv
reports/kpi/kpi_2017Q1.csv
reports/kpi/kpi_2018Q1.csv
reports/kpi/kpi_2017Q2.csv
reports/kpi/kpi_2017Q4.csv
reports/kpi/kpi_2017Q3.csv
reports/kpi/kpi_2016Q4.csv
reports/kpi/kpi_2018Q2.csv
reports/kpi/kpi_2016Q2.csv
reports/kpi/kpi_2016Q3.csv
reports/kpi/kpi_all_periods.csv
reports/kpi/kpi_2018Q3.csv
reports/kpi/kpi_2016Q1.csv
reports/kpi/kpi_2018Q4.csv
reports/contact_lists/contact_2018Q2.csv
reports/contact_lists/contact_2018Q4.csv
reports/contact_lists/contact_2016Q3.csv
reports/contact_lists/contact_2016Q2.csv
reports/contact_lists/contact_2017Q1.csv
reports/contact_lists/contact_2016Q1.csv
reports/contact_lists/contact_2018Q3.csv
reports/contact_lists/contact_2017Q4.csv
reports/contact_lists/contact_2018Q1.csv
reports/contact_lists/contact_2017Q2.csv
reports/contact_lists/contact_2017Q3.csv
reports/contact_lists/contact_2015Q4.csv
reports/contact_lists/contact_2016Q4.csv
reports/segments/customer_de